In [1]:
import json
with open("r4_samples.json", "r", encoding="utf-8") as f:
    r0_samples = json.load(f)

In [2]:
print(len(r0_samples))

5279


In [3]:
r0_samples[2]

{'q*': 'Please select the correct option(s) from the following options given the question:\nQuestion: Which of the following Panasonic products can operate in an environment with a maximum temperature of 60°C and also feature ball bearings?\nOptions:\nA. Grid-EYE High Gain Type infrared array sensor\nB. ASEN80211 AC Fan Motor\nC. EKM-B Passive Infrared Sensor\nD. AMN Passive Infrared Sensor\nE. EKM-C Passive Infrared Sensor\nYour output must strictly follow this format:\n{"answer": <the list of selected options, e.g., ["A", "B", "C", "D", "E"]>}',
 'a*': '{"answer": ["B"]}',
 'options*': ['A. Grid-EYE High Gain Type infrared array sensor',
  'B. ASEN80211 AC Fan Motor',
  'C. EKM-B Passive Infrared Sensor',
  'D. AMN Passive Infrared Sensor',
  'E. EKM-C Passive Infrared Sensor'],
 'documents': ['**Panasonic**\nideas for life\n\nAC Fan Motor\n\n80 sq.×25t\n(ASEN8)\n\nDIMENSIONS (mm inch)\n\nRoHS Directive compatibility information\nhttp://www.nais-e.com/\n\nRATING\n\nLead wire type, St

In [4]:
import re
import string

def has_full_lettered_options(sample, min_options=2):
    """
    Detect if options are in the options* field as full options.
    Returns True if options* contains entries that look like complete options
    (with identifier and text).

    Handles various formats:
    - Standard lettered: "A. text", "B) text", "C: text"
    - Numbered: "1. text", "2) text"
    - Mixed/special: "#2 combination relay..."
    - Simple identifiers: "A text" (without punctuation)
    """
    options = sample.get("options*", [])

    if not options or len(options) < min_options:
        return False

    full_option_count = 0

    for opt in options:
        if not isinstance(opt, str):
            continue

        opt = opt.strip()

        # Check if it's just a single identifier (like "A", "B", "1", "#2")
        if len(opt) <= 5 and re.match(r'^[A-Z0-9#]+$', opt):
            continue

        pattern1 = re.compile(r'^[A-Z0-9#]+[\.\)\:]?\s+.+')
        pattern2 = re.compile(r'^([A-Z0-9#]+)\s+[A-Za-z].+')

        words = opt.split()
        if len(words) >= 2 and any(word[0].isalpha() for word in words[1:] if word):
            full_option_count += 1
        elif pattern1.match(opt) or pattern2.match(opt):
            full_option_count += 1

    return full_option_count >= min_options and full_option_count >= len(options) * 0.6

In [5]:
import re
import string

def has_embedded_options_in_q(sample, min_options=2):
    """
    Detect if options are embedded in the question string.
    Returns True if the question contains lettered options (A, B, C, etc.)
    with at least min_options distinct options.
    """
    q = sample.get("q*", "")
    if not q:
        return False

    # ---------- 1. MULTILINE OPTIONS ----------
    multiline_pattern = re.compile(r'(?m)^\s*([A-Z])[\.\)\:]?\s+.+$')
    multiline_matches = multiline_pattern.findall(q)
    if len(set(multiline_matches)) >= min_options:
        if len(multiline_matches) >= min_options:
            letters = sorted(set(multiline_matches))
            expected = list(string.ascii_uppercase[:len(letters)])
            if letters == expected:
                return True

    # ---------- 2. INLINE OPTIONS WITH "Options:" MARKER ----------
    if "Options:" in q or "options:" in q:
        options_match = re.search(r'(?:[Oo]ptions:|\n)\s*(.+?)(?:\n\n|\n[A-Z]\.|\n\d\.|$)',
                                 q, re.DOTALL | re.IGNORECASE)

        if options_match:
            options_text = options_match.group(1)
            pattern = re.compile(r'([A-Z])[\.\)\:]?\s+(.+?)(?=\s+[A-Z][\.\)\:]?\s+|\s*$|\.\s*$)',
                               re.DOTALL)

            matches = []
            pos = 0
            while pos < len(options_text):
                match = pattern.match(options_text[pos:])
                if match:
                    matches.append(match.group(1))
                    pos += match.end()
                else:
                    pos += 1

            if len(set(matches)) >= min_options:
                letters = sorted(set(matches))
                expected = list(string.ascii_uppercase[:len(letters)])
                if letters == expected:
                    return True

    # ---------- 3. DIRECT PATTERN MATCH IN ENTIRE QUESTION ----------
    letter_patterns = [
        r'([A-Z])[\.\)\:]?\s+[^A-Z]*?(?=\s+[A-Z][\.\)\:]?\s+|\s*$)',
        r'\b([A-Z])[\.\)\:]\s+',
        r'\(\s*([A-Z])\s*\)\s+'
    ]

    all_matches = []
    for pattern_str in letter_patterns:
        matches = re.findall(pattern_str, q, re.IGNORECASE)
        all_matches.extend(matches)

    if len(set(all_matches)) >= min_options:
        letters = sorted(set(all_matches))
        if len(letters) >= min_options:
            expected = list(string.ascii_uppercase[:max(5, len(letters))])
            found_in_expected = [l for l in letters if l in expected[:len(letters)+2]]
            if len(found_in_expected) >= min_options:
                return True

    # ---------- 4. CHECK IF options* FIELD IS JUST LETTERS ----------
    options_list = sample.get("options*", [])
    if options_list and len(options_list) >= min_options:
        all_single_letters = all(len(str(opt).strip()) == 1 and
                                str(opt).strip().isalpha() and
                                str(opt).strip().isupper()
                                for opt in options_list)
        if all_single_letters:
            letters_in_q = re.findall(r'([A-Z])[\.\)\:]?\s+[^A-Z]{10,}', q)
            if len(set(letters_in_q)) >= min_options:
                return True

    return False

In [6]:
import re

def embed_options_into_question(sample):
    """
    Embed options from options* field into the q* field.
    Returns a new sample with embedded options.
    """
    if not sample.get('options*'):
        return sample

    q = sample.get('q*', '')
    options = sample.get('options*', [])

    if re.search(r'[Oo]ptions:\s*([A-Z0-9#])', q):
        return sample

    embedded_options = "Options: "
    for i, opt in enumerate(options):
        if isinstance(opt, str):
            opt = opt.strip()

            if re.match(r'^([A-Z])\s+', opt):
                embedded_options += opt
            elif re.match(r'^([A-Z])[\.\)\:]', opt):
                embedded_options += opt
            elif re.match(r'^[A-Z]$', opt):
                embedded_options += opt
            else:
                identifier = chr(ord('A') + i) if i < 26 else str(i)
                embedded_options += f"{identifier}. {opt}"

            if i < len(options) - 1:
                embedded_options += " "

    new_q = q.rstrip()
    if not new_q.endswith(('?', '.', '!')):
        new_q += '.'
    new_q += " " + embedded_options

    new_sample = sample.copy()
    new_sample['q*'] = new_q

    return new_sample

In [7]:
# Step 1: samples that already have options embedded in the question text.
# These should NOT also be routed through filter1 / embedding.
r0_samples_filter2 = [item for item in r0_samples if has_embedded_options_in_q(item)]
filter2_ids = {id(item) for item in r0_samples_filter2}
print(f"filter2 (already embedded): {len(r0_samples_filter2)}")

filter2 (already embedded): 5279


In [8]:
# Step 2: only look for "full lettered options in options*" among samples
# NOT already caught by filter2.
remaining_after_f2 = [item for item in r0_samples if id(item) not in filter2_ids]

r0_samples_filter1 = [item for item in remaining_after_f2 if has_full_lettered_options(item)]
filter1_ids = {id(item) for item in r0_samples_filter1}
print(f"filter1 (needs embedding): {len(r0_samples_filter1)}")

filter1 (needs embedding): 0


In [9]:
# Step 3: whatever is left after both filters — samples with no usable
# lettered-option structure detected at all.
remain_r1 = [item for item in remaining_after_f2 if id(item) not in filter1_ids]
print(f"remain (unmatched by either filter): {len(remain_r1)}")

remain (unmatched by either filter): 0


In [10]:
# Sanity check — these three buckets must now partition r0_samples exactly.
total = len(r0_samples_filter1) + len(r0_samples_filter2) + len(remain_r1)
print(f"r0_samples: {len(r0_samples)}  |  sum of buckets: {total}")
assert total == len(r0_samples), "Buckets are not a true partition — investigate before continuing!"

r0_samples: 5279  |  sum of buckets: 5279


In [11]:
remain_r1[0] if remain_r1 else print("No unmatched samples.")

No unmatched samples.


In [12]:
r1_samples_filter1 = [embed_options_into_question(sample) for sample in r0_samples_filter1]

In [13]:
merged_list = [*r1_samples_filter1, *r0_samples_filter2]
print(f"Final merged sample count (filter1 + filter2, no overlap): {len(merged_list)}")

Final merged sample count (filter1 + filter2, no overlap): 5279


In [14]:
with open('r4_samples_filter.json', 'w', encoding='utf-8') as f:
    json.dump(merged_list, f, ensure_ascii=False, indent=2)